# ETL fundos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta
import io
import zipfile

In [2]:
ano_mes = "202508"
caminho_fundos = Path("fundos")

caminho_fundos.mkdir(exist_ok=True)

# while ano_mes != "202608":
#     response = requests.get(rf"https://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/cda_fi_{ano_mes}.zip", stream=True)

#     if response.status_code == 200:
#         with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
#             arquivos = zip_ref.namelist()
            
#             bloco_2_lista= [arq for arq in arquivos if "BLC_2" in arq]

#             if bloco_2_lista:
#                 bloco_2 = bloco_2_lista[0]
#                 zip_ref.extract(bloco_2, path=caminho_fundos)
#                 print(f"{bloco_2} baixado com sucesso\n")
#             else:
#                 print(f"nenhum arquivo relacionado foi encontrado em {ano_mes}")
#     else:
#         print(f"ERRO: falha no mÊs {ano_mes}: {response.status_code}")
#     ano_mes = datetime.strptime(ano_mes, "%Y%m")

#     ano_mes += relativedelta(months=1)

#     ano_mes = ano_mes.strftime('%Y%m')    
            

In [3]:
df_fundos_def = pd.DataFrame()
for fundo in caminho_fundos.glob('*.csv'):
    df_fundo = pd.read_csv(fundo, encoding='latin1', sep=';', dtype={'TP_NEGOC': str})
    cols_importantes = ["DT_COMPTC", "CNPJ_FUNDO_CLASSE", "DENOM_SOCIAL","TP_APLIC", "CNPJ_FUNDO_CLASSE_COTA", "TP_ATIVO","NM_FUNDO_CLASSE_SUBCLASSE_COTA", "VL_MERC_POS_FINAL"]
    df_fundo = df_fundo[cols_importantes]
    df_fundo = df_fundo[df_fundo['VL_MERC_POS_FINAL'] >= 100000]
    df_fundos_def = pd.concat([df_fundos_def, df_fundo], axis=0, ignore_index=True)

df_fundos_def.shape

(1203043, 8)

In [4]:
df_fundos_def['DT_COMPTC'].value_counts()

DT_COMPTC
2025-12-31    114015
2026-01-31    113864
2025-08-31    113772
2025-09-30    113732
2025-11-30    113015
2025-10-31    112823
2026-03-31    112360
2026-02-28    112030
2026-04-30    109511
2026-05-31     71343
2026-06-30     69870
2026-07-31     46708
Name: count, dtype: int64

# DF -> Grafo

In [5]:
import networkx as nx

In [6]:
df_fundos_def['DT_COMPTC'] = pd.to_datetime(df_fundos_def['DT_COMPTC'])

In [7]:
resultados_centralidade = []

"""
esse for irá "fatiar" o df pelos meses
a var mes vai guardar o "rótulo" do mês (ex: 05-2026)
dados_mes é o df em si
"""
for mes, dados_mes in df_fundos_def.groupby(df_fundos_def['DT_COMPTC'].dt.to_period('M')):

    # construção do grafo direcional
    G_mes = nx.from_pandas_edgelist(
        dados_mes,
        source='CNPJ_FUNDO_CLASSE',
        target="CNPJ_FUNDO_CLASSE_COTA",
        edge_attr="VL_MERC_POS_FINAL",
        create_using=nx.DiGraph()
    )

    # cálculo da centralidade de grau (métrica Efeito Manada)
    in_degree = nx.in_degree_centrality(G_mes)

    # Métrica de Risco de Contágio (Quem distribui investimento)
    out_degree = nx.out_degree_centrality(G_mes)

    # armazenando os resultados
    for cnpj in G_mes.nodes():
        resultados_centralidade.append({
            "Mes" : str(mes),
            "CNPJ" : cnpj,
            "Central_Grau_Ent" : in_degree[cnpj],
            "Central_Grau_Saida" : out_degree[cnpj]
        })

In [8]:
# matriz de features
df_features = pd.DataFrame(resultados_centralidade)
df_features = df_features.sort_values(by=['CNPJ', 'Mes']).reset_index(drop=True)

In [9]:
df_features.head(10)

,Mes,CNPJ,Central_Grau_Ent,Central_Grau_Saida
0,2025-08,00.068.305/0001-35,0.0,0.000038
1,2025-09,00.068.305/0001-35,0.0,0.000037
2,2025-10,00.068.305/0001-35,0.0,0.000038
3,2025-11,00.068.305/0001-35,0.0,0.000037
4,2025-12,00.068.305/0001-35,0.0,0.000037
5,2026-01,00.068.305/0001-35,0.0,0.000037
6,2026-02,00.068.305/0001-35,0.0,0.000038
7,2026-03,00.068.305/0001-35,0.0,0.000038
8,2026-04,00.068.305/0001-35,0.0,0.000038
9,2026-05,00.068.305/0001-35,0.0,0.000048


# Modelagem ML

In [10]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

In [11]:
# transformando meses em colunas
df_ml = df_features.pivot(index='CNPJ',
                          columns='Mes',
                          values=['Central_Grau_Ent', 'Central_Grau_Saida']).fillna(0)

df_ml.columns = [f"{metrica}_{mes}" for metrica, mes in df_ml.columns]

df_ml.shape

(30449, 24)

In [28]:
# separando os dfs para cada caso
colunas_manada = [col for col in df_ml.columns if 'Ent' in col]
colunas_contagio = [col for col in df_ml.columns if 'Saida' in col]

df_manada = df_ml[colunas_manada]
df_contagio = df_ml[colunas_contagio]
print(f"Tamanho DF de efeito manada: {len(df_manada)}")
print(f'Tamanho DF risco de contágio: {len(df_contagio)}')

Tamanho DF de efeito manada: 30449
Tamanho DF risco de contágio: 30449


In [29]:
df_contagio = df_contagio.loc[~(df_contagio == 0).all(axis=1)]
df_manada = df_contagio.loc[~(df_contagio == 0).all(axis=1)]

print(f"Tamanho DF de efeito manada: {len(df_manada)}")
print(f'Tamanho DF risco de contágio: {len(df_contagio)}')

Tamanho DF de efeito manada: 24354
Tamanho DF risco de contágio: 24354


## Efeito Manada

In [30]:
# Isolation Forest
modelo_if = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
# LOF (Local Outlier Factor)
modelo_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
# One-Class SVM
modelo_svm = OneClassSVM(nu=0.01, kernel='rbf', gamma='scale')



df_manada['Previsao_if'] = modelo_if.fit_predict(df_manada)
df_manada['Previsao_lof'] = modelo_lof.fit_predict(df_manada)
df_manada['Previsao_svm'] = modelo_svm.fit_predict(df_manada)

In [31]:
df_manada[['Previsao_if', 'Previsao_lof', 'Previsao_svm']].value_counts()

Previsao_if  Previsao_lof  Previsao_svm
 1            1            -1              18487
                            1               5379
             -1             1                244
-1            1            -1                174
                            1                 70
Name: count, dtype: int64

### Comitê de riscos

In [32]:
# O código verifica quem deu -1, transforma em True/False e soma (True vale 1)
df_manada['Votos_Anomalia'] = (df_manada[['Previsao_if', 'Previsao_lof', 'Previsao_svm']] == -1).sum(axis=1)

df_manada['votos_if_svm'] = (df_manada[['Previsao_if', 'Previsao_svm']] == -1).sum(axis=1)

df_manada['votos_if_lof'] = (df_manada[['Previsao_if', 'Previsao_lof']] == -1).sum(axis=1)

df_manada['votos_lof_svm'] = (df_manada[['Previsao_lof', 'Previsao_svm']] == -1).sum(axis=1)
# Votação por Maioria (Pelo menos 2 modelos concordam)
df_risco_maioria = df_manada[df_manada['Votos_Anomalia'] >= 2]

# Consenso Absoluto (Os 3 modelos concordam)
df_risco_absoluto = df_manada[df_manada['Votos_Anomalia'] == 3]

df_risco_if_svm = df_manada[df_manada['votos_if_svm'] == 2]

df_risco_if_lof = df_manada[df_manada['votos_if_lof'] == 2]

df_risco_lof_svm = df_manada[df_manada['votos_lof_svm'] == 2]

print("--- Apuração do Comitê de Risco ---")
print(f"Anomalias validadas por Maioria (>= 2 votos): {len(df_risco_maioria)}")
print(f"Anomalias extremas validadas por Consenso (3 votos): {len(df_risco_absoluto)}")
print(f"Anomalias validadas por IF e SVM: {len(df_risco_if_svm)}")
print(f"Anomalias validadas por IF e LOF: {len(df_risco_if_lof)}")
print(f"Anomalias validadas por LOF e SVM: {len(df_risco_lof_svm)}")


--- Apuração do Comitê de Risco ---
Anomalias validadas por Maioria (>= 2 votos): 174
Anomalias extremas validadas por Consenso (3 votos): 0
Anomalias validadas por IF e SVM: 174
Anomalias validadas por IF e LOF: 0
Anomalias validadas por LOF e SVM: 0


## Risco de contágio

In [33]:
modelo_if_cont = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
modelo_lof_cont = LocalOutlierFactor(contamination=0.01, n_neighbors=20)
modelo_svm_cont = OneClassSVM(nu=0.01, kernel='rbf', gamma='scale')

df_contagio['Previsao_if'] = modelo_if_cont.fit_predict(df_contagio)
df_contagio['Previsao_lof'] = modelo_lof_cont.fit_predict(df_contagio)
df_contagio['Previsao_svm'] = modelo_svm_cont.fit_predict(df_contagio)

In [34]:
df_contagio['Votos_anomalia'] = (df_contagio[['Previsao_if', 'Previsao_lof', 'Previsao_svm']] == -1).sum(axis=1)

df_voto_absoluto_cont = df_contagio[df_contagio['Votos_anomalia'] == 3]
df_voto_maioria_cont = df_contagio[df_contagio['Votos_anomalia'] >= 2]

df_contagio['Votos_anomalia_if_lof'] = (df_contagio[['Previsao_if', 'Previsao_lof']] == -1).sum(axis=1)
df_voto_if_lof_cont = df_contagio[df_contagio['Votos_anomalia_if_lof'] == 2]

df_contagio['Votos_anomalia_if_svm'] = (df_contagio[['Previsao_if', 'Previsao_svm']] == -1).sum(axis=1)
df_voto_if_svm_cont = df_contagio[df_contagio['Votos_anomalia_if_svm'] == 2]

df_contagio['Votos_anomalia_lof_svm'] = (df_contagio[['Previsao_lof', 'Previsao_svm']] == -1).sum(axis=1)
df_voto_lof_svm_cont = df_contagio[df_contagio['Votos_anomalia_lof_svm'] == 2]

votos_lof = (df_contagio['Previsao_lof'] == -1).sum()
votos_if = (df_contagio['Previsao_if'] == -1).sum()
votos_svm = (df_contagio['Previsao_svm'] == -1).sum()

print(f'Quantidade de votos absolutos: {len(df_voto_absoluto_cont)}')
print(f'Quantidade de votos maioria: {len(df_voto_maioria_cont)}')
print(f'Anomalias validadas por IF e LOF: {len(df_voto_if_lof_cont)}')
print(f'Anomalias validadas por IF e SVM: {len(df_voto_if_svm_cont)}')
print(f'Anomalias validadas por LOF e SVM: {len(df_voto_lof_svm_cont)}')
print(f'Anomalias validadas apenas por IF: {votos_if}')
print(f'Anomalias validadas apenas por LOF: {votos_lof}')
print(f'Anomalias validadas apenas por SVM: {votos_svm}')

Quantidade de votos absolutos: 0
Quantidade de votos maioria: 174
Anomalias validadas por IF e LOF: 0
Anomalias validadas por IF e SVM: 174
Anomalias validadas por LOF e SVM: 0
Anomalias validadas apenas por IF: 244
Anomalias validadas apenas por LOF: 244
Anomalias validadas apenas por SVM: 18661
